# Data preprocessing and corpus aggregation
- Data preprocessing
    - packages and dependencies
    - import raw data
    - filtering (CVs; file formats `.doc`, `.pdf`, and `.docx`)
    - text extraction, normalization, standardization
    - text from corpus table to actual `.txt`
- Data aggregation
    - exploratory aggregation (for EDA)
    - corpus table
    - corpus table quality checks


## Packages


In [1]:
!pip install -q pymupdf python-docx pandas

!apt-get update -qq
!apt-get install -y -qq libreoffice

!soffice --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-opensymbol.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../000-fonts-opensymbol_2%3a102.12+LibO7.3.7-0ubuntu0.22.04.10_all.deb ...
Unpacking fonts-opensymbol (2:102.12+LibO7.3.7-0ubuntu0.22.04.10) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../001-libreoffice-style-colibre_1%3a7.3.7-0ubuntu0.22.04.10_all.deb ...
Unpacking libreoffice-style-colibre (1:7.3.7-0ubuntu0.22.04.10) ...
Selecting previously unselected package libuno-sal3.
Pre

In [2]:
import pandas as pd
import numpy as np
import fitz ## PyMuPDF
import re
import hashlib
import json
import time
import subprocess
import matplotlib.pyplot as plt

from pathlib import Path
from docx import Document
from google.colab import drive
from IPython.display import display, HTML

## Import and locate data


In [3]:
## imports
import os
import shutil
from pathlib import Path
from google.colab import drive

## safe mount helper
mount_point = "/content/drive"

## if drive is already mounted, skip remount
if os.path.ismount(mount_point):
    print(f"already mounted at {mount_point}")
else:
    ## if mount folder exists and is not empty, remove it first
    if os.path.exists(mount_point) and os.path.isdir(mount_point):
        if os.listdir(mount_point):
            shutil.rmtree(mount_point)

    ## remount fresh
    drive.mount(mount_point, force_remount=True)

## define root path
BASE = Path("/content/drive/MyDrive/data_youthnex")
print("BASE exists:", BASE.exists())

Mounted at /content/drive
BASE exists: True


Find appendix folders and files by year

In [4]:
## appendix folder + "final" OR "lb" file aggregation
rows = []
folder_hits = []

## constants
APPENDIX_FOLDER_KEYWORD = "appendices"

## process loop
for y in range(2020, 2026):
    year = str(y)
    year_path = BASE / year

    ## missing year directory check
    if not year_path.exists():
        rows.append({"year": year, "status": "year_missing", "file_path": None, "ext": None})
        folder_hits.append({
            "year": year,
            "appendix_folders_found": 0,
            "target_files_found": 0
        })
        continue

    ## find appendix folders
    appendix_matches = [
        p for p in year_path.rglob("*")
        if p.is_dir() and APPENDIX_FOLDER_KEYWORD in p.name.strip().lower()
    ]

    ## count target files ("final" OR "lb")
    target_files = []
    for folder in appendix_matches:
        for f in folder.rglob("*"):
            if f.is_file():
                name_lower = f.name.lower()
                if "final" in name_lower or "lb" in name_lower:
                    target_files.append(f)

    folder_hits.append({
        "year": year,
        "appendix_folders_found": len(appendix_matches),
        "target_files_found": len(target_files)
    })

    ## missing appendix folder check
    if not appendix_matches:
        rows.append({"year": year, "status": "no_appendix_folder_found", "file_path": None, "ext": None})
        continue

    ## appendix folder exists but no matching files
    if not target_files:
        rows.append({"year": year, "status": "no_target_files_found", "file_path": None, "ext": None})
        continue

    ## record files
    for f in target_files:
        ext = f.suffix.lower() if f.suffix else "(no_ext)"
        rows.append({
            "year": year,
            "status": "target_file_found",
            "file_path": str(f),
            "ext": ext
        })

In [5]:
folder_hits

[{'year': '2020', 'appendix_folders_found': 0, 'target_files_found': 0},
 {'year': '2021', 'appendix_folders_found': 0, 'target_files_found': 0},
 {'year': '2022', 'appendix_folders_found': 0, 'target_files_found': 0},
 {'year': '2023', 'appendix_folders_found': 1, 'target_files_found': 19},
 {'year': '2024', 'appendix_folders_found': 1, 'target_files_found': 21},
 {'year': '2025', 'appendix_folders_found': 2, 'target_files_found': 26}]

## Exploratory aggregation (useful for EDA)

In [6]:
## create yearly summary table
hits_df = pd.DataFrame(folder_hits)

hits_df["appendix_folder_found"] = hits_df["appendix_folders_found"] > 0

hits_df

,year,appendix_folders_found,target_files_found,appendix_folder_found
0,2020,0,0,False
1,2021,0,0,False
2,2022,0,0,False
3,2023,1,19,True
4,2024,1,21,True
5,2025,2,26,True


In [7]:
## create file-level appendix table
df_appendix = pd.DataFrame(rows)

## keep only actual matched files
df_appendix = df_appendix[df_appendix["status"] == "target_file_found"].copy()

## add file name
df_appendix["file_name"] = df_appendix["file_path"].apply(lambda x: Path(x).name if pd.notna(x) else None)

df_appendix.head()

,year,status,file_path,ext,file_name
3,2023,target_file_found,/content/drive/MyDrive/data_youthnex/2023/appe...,.docx,Faculty & Researcher Grant Funding 7-21-23 lb....
4,2023,target_file_found,/content/drive/MyDrive/data_youthnex/2023/appe...,.pdf,2. Faculty & Researcher Grant Funding 7-21-23 ...
5,2023,target_file_found,/content/drive/MyDrive/data_youthnex/2023/appe...,.docx,Faculty & Researcher Scholarly Work 8-23-23 lb...
6,2023,target_file_found,/content/drive/MyDrive/data_youthnex/2023/appe...,.pdf,3. Faculty & Researcher Scholarly Work 8-23-23...
7,2023,target_file_found,/content/drive/MyDrive/data_youthnex/2023/appe...,.docx,Faculty & Researcher Presentations 8-23-23 lb....


In [8]:
## imports for dedup
import re
from pathlib import Path

## clean filename down to a shared core title
def make_core_title(name: str) -> str:
    if not isinstance(name, str):
        return ""

    s = Path(name).stem.lower().strip()

    ## remove leading numbering like "1. " or "10. "
    s = re.sub(r"^\d+\.\s*", "", s)
    s = re.sub(r"^\d+\s*-\s*", "", s)
    s = re.sub(r"^\d+\s+", "", s)

    ## normalize symbols
    s = s.replace("&", "and")

    ## remove common trailing tags
    s = re.sub(r"\bfinal\b", "", s)
    s = re.sub(r"\blb\b", "", s)

    ## remove dates like 5-30-25, 05-30-2025, 3-3-24, etc.
    s = re.sub(r"\b\d{1,2}-\d{1,2}-\d{2,4}\b", "", s)

    ## collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s

## rebuild appendix table
df_appendix = pd.DataFrame(rows)
df_appendix = df_appendix[df_appendix["status"] == "target_file_found"].copy()

df_appendix["file_name"] = df_appendix["file_path"].apply(
    lambda x: Path(x).name if pd.notna(x) else None
)

## make core title
df_appendix["core_title"] = df_appendix["file_name"].apply(make_core_title)

## prefer pdf over docx over doc
ext_priority = {
    ".pdf": 1,
    ".docx": 2,
    ".doc": 3
}
df_appendix["ext_rank"] = df_appendix["ext"].map(ext_priority).fillna(99)

## sort so preferred file comes first
df_appendix = df_appendix.sort_values(
    ["year", "core_title", "ext_rank", "file_name"]
).copy()

## deduplicate within year by core title
df_appendix_dedup = df_appendix.drop_duplicates(
    subset=["year", "core_title"],
    keep="first"
).copy()

print("before dedup:", len(df_appendix))
print("after dedup:", len(df_appendix_dedup))
print("removed:", len(df_appendix) - len(df_appendix_dedup))

before dedup: 66
after dedup: 29
removed: 37


In [9]:
## table of leftover files with paths
leftover_table = (
    df_appendix_dedup[["year", "file_name", "file_path"]]
    .sort_values(["year", "file_name"])
    .reset_index(drop=True)
)

leftover_table

,year,file_name,file_path
0,2023,2. Faculty & Researcher Grant Funding 7-21-23 ...,/content/drive/MyDrive/data_youthnex/2023/appe...
1,2023,3. Faculty & Researcher Scholarly Work 8-23-23...,/content/drive/MyDrive/data_youthnex/2023/appe...
2,2023,4. Faculty & Researcher Presentations 8-23-23 ...,/content/drive/MyDrive/data_youthnex/2023/appe...
3,2023,5. Faculty & Researcher Awards & Recognition 7...,/content/drive/MyDrive/data_youthnex/2023/appe...
4,2023,6. Faculty & Researcher Local & State Service ...,/content/drive/MyDrive/data_youthnex/2023/appe...
5,2023,7. Faculty & Researcher National & Internation...,/content/drive/MyDrive/data_youthnex/2023/appe...
6,2023,8. Postdoc 7-21-23 lb.pdf,/content/drive/MyDrive/data_youthnex/2023/appe...
7,2023,9. Grad students 7-21-23 lb.pdf,/content/drive/MyDrive/data_youthnex/2023/appe...
8,2023,External Faculty 3-3-24 lb.pdf,/content/drive/MyDrive/data_youthnex/2023/appe...
9,2024,1. External Faculty final.pdf,/content/drive/MyDrive/data_youthnex/2024/appe...


In [10]:
## count leftover (deduplicated) files per year
counts_df = (
    df_appendix_dedup
    .groupby("year")
    .size()
    .reset_index(name="dedup_file_count")
    .sort_values("year")
)

counts_df

,year,dedup_file_count
0,2023,9
1,2024,10
2,2025,10


## Filtering, extraction, normalization, standardization

### Setup/configuration


In [11]:
## count leftover .doc files only
leftover_doc_df = (
    df_appendix_dedup[
        df_appendix_dedup["file_path"].apply(
            lambda x: Path(x).suffix.lower() == ".doc" if pd.notna(x) else False
        )
    ]
    .copy()
)

print("Leftover .doc files found:", len(leftover_doc_df))

leftover_doc_df[["year", "file_name", "file_path"]].sort_values(
    ["year", "file_name"]
).reset_index(drop=True)

Leftover .doc files found: 0


,year,file_name,file_path


### Legacy .doc to .docx

In [12]:
import subprocess
from pathlib import Path

BASE_CONVERTED = Path("/content/drive/MyDrive/data_youthnex_docx_converted")
BASE_CONVERTED.mkdir(parents=True, exist_ok=True)

BASE_CONVERTED.mkdir(parents=True, exist_ok=True)

## conversion function (unchanged)
def convert_doc_to_docx(src_doc: Path, dst_docx: Path) -> tuple[bool, str]:
    try:
        dst_docx.parent.mkdir(parents=True, exist_ok=True)

        if dst_docx.exists() and dst_docx.stat().st_mtime >= src_doc.stat().st_mtime:
            return True, "skipped_up_to_date"

        cmd = [
            "soffice", "--headless", "--nologo", "--nofirststartwizard",
            "--convert-to", "docx",
            "--outdir", str(dst_docx.parent),
            str(src_doc),
        ]
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        expected = dst_docx.parent / (src_doc.stem + ".docx")

        if r.returncode != 0:
            return False, f"convert_error: {r.stderr.strip()[:300]}"
        if not expected.exists():
            return False, "convert_error: output_missing"

        if expected.resolve() != dst_docx.resolve():
            expected.replace(dst_docx)

        return True, "converted"

    except Exception as e:
        return False, f"convert_error: {str(e)}"


## isolate leftover appendix .doc files
leftover_doc_df = df_appendix_dedup[df_appendix_dedup["ext"] == ".doc"].copy()


## initialize counters
converted = skipped = failed = 0
fail_samples = []


## process only leftover appendix .doc files
for _, row in leftover_doc_df.iterrows():

    src_doc = Path(row["file_path"])

    if not src_doc.exists():
        failed += 1
        if len(fail_samples) < 5:
            fail_samples.append((str(src_doc), "file_missing"))
        continue

    ## preserve folder structure relative to BASE
    rel = src_doc.relative_to(BASE)
    dst_docx = (BASE_CONVERTED / rel).with_suffix(".docx")

    ## conversion
    ok, msg = convert_doc_to_docx(src_doc, dst_docx)

    if ok and msg == "converted":
        converted += 1
    elif ok and msg == "skipped_up_to_date":
        skipped += 1
    else:
        failed += 1
        if len(fail_samples) < 5:
            fail_samples.append((str(src_doc), msg))


## print summary
print(f"APPENDIX DOC→DOCX: converted={converted}, skipped_up_to_date={skipped}, failed={failed}")

if fail_samples:
    print("Sample failures:")
    for p, m in fail_samples:
        print(" ", p, "=>", m)

APPENDIX DOC→DOCX: converted=0, skipped_up_to_date=0, failed=0


### Extraction

In [14]:
ALLOWED_EXTS = {".pdf", ".docx"}

## extract text from pdf using PyMuPDF (fitz)
def extract_text_pdf(path: Path, block_gap_threshold=14):
    try:
        output_pages = []

        with fitz.open(path) as doc:
            for page in doc:
                text_dict = page.get_text("dict")
                extracted_lines = []

                ## collect text lines from text blocks only
                for block in text_dict["blocks"]:
                    if block["type"] != 0:
                        continue

                    for line in block["lines"]:
                        spans = line["spans"]
                        line_text = "".join(span["text"] for span in spans).strip()

                        if not line_text:
                            continue

                        x0 = min(span["bbox"][0] for span in spans)
                        y0 = min(span["bbox"][1] for span in spans)
                        x1 = max(span["bbox"][2] for span in spans)
                        y1 = max(span["bbox"][3] for span in spans)

                        extracted_lines.append({
                            "text": line_text,
                            "x0": x0,
                            "y0": y0,
                            "x1": x1,
                            "y1": y1,
                        })

                ## sort lines in reading order
                extracted_lines.sort(key=lambda ln: (round(ln["y0"], 1), ln["x0"]))

                if not extracted_lines:
                    output_pages.append("")
                    continue

                page_parts = [extracted_lines[0]["text"]]
                prev_line = extracted_lines[0]

                for line in extracted_lines[1:]:
                    vertical_gap = line["y0"] - prev_line["y1"]

                    ## larger vertical gap = likely new block/section
                    if vertical_gap > block_gap_threshold:
                        page_parts.append("\n\n")
                    else:
                        page_parts.append("\n")

                    page_parts.append(line["text"])
                    prev_line = line

                output_pages.append("".join(page_parts).strip())

        return True, "\n\n".join(output_pages).strip(), None

    except Exception as e:
        return False, None, f"pdf_error: {str(e)}"


## extract text from docx using python-docx
def extract_text_docx(path: Path):
    try:
        doc = Document(path)
        text = "\n".join(p.text for p in doc.paragraphs)
        return True, text, None
    except Exception as e:
        return False, None, f"docx_error: {str(e)}"


## route file based on file extension
def extract_text_any(path: Path):
    ext = path.suffix.lower()
    if ext == ".pdf":
        return extract_text_pdf(path)
    if ext == ".docx":
        return extract_text_docx(path)

    ## failure if file type is not .pdf or .docx
    return False, None, f"unsupported_ext: {ext}"


## normalize for single-line .txt format
def normalize_text_for_txt(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
    text = " ".join(text.split())
    return text


## initialize rows, seen_doc_ids
rows = []
seen_doc_ids = set()


## scan appendix file table and extract text
def extract_from_appendix_table(df_files: pd.DataFrame):
    """
    df_files: deduplicated appendix file table
              should contain at least year, file_path, ext, and file_name
    """

    for _, row in df_files.iterrows():
        f = Path(row["file_path"])

        if not f.exists():
            rows.append({
                "year": row["year"],
                "doc_id": str(row["file_path"]),
                "path": str(row["file_path"]),
                "ext": row["ext"],
                "ok": False,
                "fail_reason": "file_missing",
                "n_chars": 0,
                "n_words": 0,
                "text": None
            })
            continue

        ## only process extractable types
        if f.suffix.lower() not in ALLOWED_EXTS:
            rows.append({
                "year": row["year"],
                "doc_id": str(row["file_path"]),
                "path": str(row["file_path"]),
                "ext": f.suffix.lower(),
                "ok": False,
                "fail_reason": f"unsupported_ext: {f.suffix.lower()}",
                "n_chars": 0,
                "n_words": 0,
                "text": None
            })
            continue

        ## unique identifier based on relative path from BASE if possible
        try:
            doc_id = str(f.relative_to(BASE))
        except Exception:
            doc_id = str(f)

        ## skip if already processed
        if doc_id in seen_doc_ids:
            continue
        seen_doc_ids.add(doc_id)

        ## extract
        ok, text, reason = extract_text_any(f)

        if ok and text:
            text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()

        rows.append({
            "year": row["year"],
            "doc_id": doc_id,
            "path": str(f),
            "ext": f.suffix.lower(),
            "ok": ok,
            "fail_reason": reason,
            "n_chars": len(text) if text else 0,
            "n_words": len(re.findall(r"\S+", text)) if text else 0,
            "text": text if ok else None
        })


## use deduplicated appendix files
extract_input_df = df_appendix_dedup.copy()

## if you converted .doc files earlier, swap them to converted .docx paths when available
def resolve_preferred_path(path_str: str) -> str:
    src = Path(path_str)

    if src.suffix.lower() == ".doc":
        try:
            rel = src.relative_to(BASE)
            converted = (BASE_CONVERTED / rel).with_suffix(".docx")
            if converted.exists():
                return str(converted)
        except Exception:
            pass

    return str(src)

extract_input_df["file_path"] = extract_input_df["file_path"].apply(resolve_preferred_path)
extract_input_df["ext"] = extract_input_df["file_path"].apply(lambda x: Path(x).suffix.lower())

## extract text from appendix files
extract_from_appendix_table(extract_input_df)

## final appendix corpus table
df_appendix_corpus = pd.DataFrame(rows)
df_appendix_corpus.head()

,year,doc_id,path,ext,ok,fail_reason,n_chars,n_words,text
0,2023,2023/appendices/External Faculty 3-3-24 lb.pdf,/content/drive/MyDrive/data_youthnex/2023/appe...,.pdf,True,None,33244,4637,"Valerie N. Adams-Bass, Rutgers University\n\nP..."
1,2023,2023/appendices/5. Faculty & Researcher Awards...,/content/drive/MyDrive/data_youthnex/2023/appe...,.pdf,True,None,3376,464,Catherine Bradshaw\n• University Professorship...
2,2023,2023/appendices/2. Faculty & Researcher Grant ...,/content/drive/MyDrive/data_youthnex/2023/appe...,.pdf,True,None,47988,6651,Nancy Deutsch\n• W.T. Grant Foundation Mixed M...
3,2023,2023/appendices/6. Faculty & Researcher Local ...,/content/drive/MyDrive/data_youthnex/2023/appe...,.pdf,True,None,20232,2882,"Nancy Deutsch\n• Dean Search Committee, Co-Cha..."
4,2023,2023/appendices/7. Faculty & Researcher Nation...,/content/drive/MyDrive/data_youthnex/2023/appe...,.pdf,True,None,8429,1173,Nancy Deutsch\n• Boys & Girls Clubs of America...


## Corpus table

In [ ]:
## build corpus table
## df_corpus = pd.DataFrame(rows)

### Corpus table quality checks

In [15]:
## examine aggregation quality
print("total rows:", len(df_appendix_corpus))
print("success vs fail:\n", df_appendix_corpus["ok"].value_counts())

## examine dataframe
print("rows:", len(df_appendix_corpus))
print("\nok vs fail:\n", df_appendix_corpus["ok"].value_counts(dropna=False))
print("\nfailure reasons:\n",
      df_appendix_corpus.loc[df_appendix_corpus["ok"] == False, "fail_reason"]
      .value_counts().head(10)
)

## preview
cols = ["year", "doc_id", "ext", "ok", "n_chars", "n_words", "fail_reason", "text"]
df_appendix_corpus.reindex(columns=cols).head(20)

total rows: 29
success vs fail:
 ok
True    29
Name: count, dtype: int64
rows: 29

ok vs fail:
 ok
True    29
Name: count, dtype: int64

failure reasons:
 Series([], Name: count, dtype: int64)


,year,doc_id,ext,ok,n_chars,n_words,fail_reason,text
0,2023,2023/appendices/External Faculty 3-3-24 lb.pdf,.pdf,True,33244,4637,None,"Valerie N. Adams-Bass, Rutgers University\n\nP..."
1,2023,2023/appendices/5. Faculty & Researcher Awards...,.pdf,True,3376,464,None,Catherine Bradshaw\n• University Professorship...
2,2023,2023/appendices/2. Faculty & Researcher Grant ...,.pdf,True,47988,6651,None,Nancy Deutsch\n• W.T. Grant Foundation Mixed M...
3,2023,2023/appendices/6. Faculty & Researcher Local ...,.pdf,True,20232,2882,None,"Nancy Deutsch\n• Dean Search Committee, Co-Cha..."
4,2023,2023/appendices/7. Faculty & Researcher Nation...,.pdf,True,8429,1173,None,Nancy Deutsch\n• Boys & Girls Clubs of America...
5,2023,2023/appendices/4. Faculty & Researcher Presen...,.pdf,True,47390,6666,None,Nancy Deutsch\nSCHOLARLY PRESENTATIONS - REFER...
6,2023,2023/appendices/3. Faculty & Researcher Schola...,.pdf,True,92540,12778,None,Nancy Deutsch\nJOURNAL ARTICLES AND MONOGRAPHS...
7,2023,2023/appendices/9. Grad students 7-21-23 lb.pdf,.pdf,True,27793,3815,None,Noor A. Alwani\nAWARDS & FUNDING\n• 2019-2023 ...
8,2023,2023/appendices/8. Postdoc 7-21-23 lb.pdf,.pdf,True,34969,4922,None,Lydia A. Beahm\nARTICLES IN PEER REVIEWED JOUR...
9,2024,2024/appendices/1. External Faculty final.pdf,.pdf,True,14392,1946,None,"Joanna Lee Williams, Search Institute\n\nBOOKS..."


In [18]:
## copy of successfully extracted documents
df_ok = df_appendix_corpus[df_appendix_corpus["ok"] == True].copy()

## random doc for quality check (any extension)
row = df_ok.sample(1, random_state=7).iloc[0]

## diagnostic of random doc
print("DOC ID:", row["doc_id"])
print("PATH:", row["path"])
print("EXT:", row["ext"])
print("CHARS:", row["n_chars"], "| WORDS:", row["n_words"])

DOC ID: 2023/appendices/2. Faculty & Researcher Grant Funding 7-21-23 lb.pdf
PATH: /content/drive/MyDrive/data_youthnex/2023/appendices/2. Faculty & Researcher Grant Funding 7-21-23 lb.pdf
EXT: .pdf
CHARS: 47988 | WORDS: 6651


In [19]:
## get extracted text from document row
text = row["text"]

## display text in scrollable html container for evaluation
display(HTML(f"""
<div style="
    height:600px;
    overflow:auto;
    border:1px solid #ccc;
    padding:10px;
    white-space:pre-wrap;
    font-family:monospace;
">
{text}
</div>
"""))

### Corpus table visualizations

In [ ]:
## box plot
## remove rows where n_words or year values are missing
plot_df = df_corpus.dropna(subset=["n_words", "year"])

## year column to string, sort
plot_df["year"] = plot_df["year"].astype(str)
years = sorted(plot_df["year"].unique())

## create list of word count arrays grouped by year
data = [plot_df.loc[plot_df["year"] == y, "n_words"] for y in years]

## plot
plt.figure(figsize=(10, 5))
plt.boxplot(data, labels=years)
plt.title("Distribution of Word Count by Year")
plt.xlabel("Year")
plt.ylabel("Word Count")
plt.tight_layout()
plt.show()

In [ ]:
## violin plot
## remove rows where n_words or year values are missing
plot_df = df_corpus.dropna(subset=["n_words", "year"])

## year column to string, sort
plot_df["year"] = plot_df["year"].astype(str)
years = sorted(plot_df["year"].unique())

## create list of word count arrays grouped by year
data = [plot_df.loc[plot_df["year"] == y, "n_words"] for y in years]

## plot
plt.figure(figsize=(10, 5))
plt.violinplot(data, showmedians=True)
plt.xticks(range(1, len(years) + 1), years)
plt.title("Distribution of Word Count by Year")
plt.xlabel("Year")
plt.ylabel("Word Count")
plt.tight_layout()
plt.show()

## Text from corpus table to actual `.txt`

In [ ]:
## output root directory
OUT_TXT_ROOT = Path("/content/drive/MyDrive/youthnex_txt")
OUT_TXT_ROOT.mkdir(parents=True, exist_ok=True)

## compute sha256 has of text if it does not exist
def _sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="ignore")).hexdigest()

## write file using temp file then replace
def _atomic_write(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(content, encoding="utf-8", errors="ignore")
    tmp.replace(path)

## initialize counters
written, skipped = 0, 0

## keep successful extractions only
df_ok = df_corpus[df_corpus["ok"] == True].copy()

## iterate through documents
for _, row in df_ok.iterrows():
    doc_id = row["doc_id"]
    text = row["text"] or ""

    ## mirror folder structure but swap to .txt
    out_txt = OUT_TXT_ROOT / Path(doc_id).with_suffix(".txt")
    out_meta = out_txt.with_suffix(".json")

    ## preserve line structure
    text = str(text).replace("\r\n","\n").replace("\r","\n").strip()

    ## text hash
    text_hash = _sha256_text(text)

    ## skip if file already exists and hash matches
    if out_txt.exists() and out_meta.exists():
        try:
            meta = json.loads(out_meta.read_text(encoding="utf-8"))
            if meta.get("text_sha256") == text_hash:
                skipped += 1
                continue
        except Exception:
            pass

    ## write text file
    _atomic_write(out_txt, text)

    ## metadata dictionary
    meta = {
        "doc_id": doc_id,
        "source_path": row["path"],
        "year": row["year"],
        "ext": row["ext"],
        "extracted_at_epoch": int(time.time()),
        "text_sha256": text_hash,
        "n_chars": int(row.get("n_chars", len(text))),
        "n_words": int(row.get("n_words", len(text.split()))),
    }

    ## metadata file
    _atomic_write(out_meta, json.dumps(meta, indent=2))

    ## increment counter
    written += 1

## summary
print(f"Wrote {written} .txt files | skipped {skipped} (already current) | total ok {len(df_ok)}")
print("Output root:", OUT_TXT_ROOT)